# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
import os
import json
import hashlib
from datetime import datetime, timezone
from typing import List, Literal, Optional, TypedDict

import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from tavily import TavilyClient
from dotenv import load_dotenv, find_dotenv

from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.tooling import tool
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run
from lib.memory import ShortTermMemory

In [3]:
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

MODEL = "gpt-4o-mini"

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Same path, collection name and embedding function as Part 1
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

# Long-term memory (stand-out): a second collection in the same persistent database
memory_collection = chroma_client.get_or_create_collection("udaplay_memory", embedding_function=embedding_fn)

print("Games in vector DB:", collection.count(), "| Long-term memories:", memory_collection.count())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Games in vector DB: 15 | Long-term memories: 0


In [ ]:
# Environment variables are loaded in the setup cell above.

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [4]:

def llm_structured(system: str, user: str, schema):
    """Ask the LLM (lib.llm.LLM) and get the reply back as an instance of the Pydantic model `schema`."""
    llm = LLM(model=MODEL, temperature=0.0)
    response = llm.invoke(
        [SystemMessage(content=system), UserMessage(content=user)],
        response_format=schema,
    )
    return schema.model_validate_json(response.content)


def call_tool(t, **kwargs):
    """Run a lib Tool and log the call."""
    shown = ", ".join(f"{k}={repr(v)[:70]}" for k, v in kwargs.items())
    print(f"   [TOOL CALL] {t.name}({shown})")
    return t(**kwargs)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# TODO: Create retrieve_game tool
# It should use the chroma client and collection you created in Part 1 (same path and name)
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, PlayStation 5, Xbox 360...
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str, n_results: int = 3) -> List[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    (plus Genre, Publisher, id and distance - a lower distance means a closer match)
    """
    res = collection.query(query_texts=[query], n_results=n_results)
    docs = []
    for i in range(len(res["ids"][0])):
        item = dict(res["metadatas"][0][i])
        item["id"] = res["ids"][0][i]
        item["distance"] = round(res["distances"][0][i], 3)
        item["source"] = "local_dataset"
        docs.append(item)
    return docs


# quick test (calls the Tool object directly)
for d in retrieve_game("Who published Grand Theft Auto: San Andreas?"):
    print("  ->", d["Name"], "|", d["Platform"], "|", d["YearOfRelease"], "|", d["Publisher"], "| dist", d["distance"])
print(retrieve_game)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  -> Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | Rockstar Games | dist 0.247
  -> Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment | dist 0.42
  -> Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment | dist 0.427
<Tool name=retrieve_game params=['query', 'n_results']>


#### Evaluate Retrieval Tool

In [6]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are enough to answer the question")
    confidence: Literal["high", "medium", "low"] = Field(description="Confidence that the docs answer the question")
    description: str = Field(description="Detailed explanation of the evaluation so an action can be taken")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> EvaluationReport:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - confidence: high / medium / low
    - description: description about the evaluation result
    """
    system = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not. "
        "Be strict: the documents must explicitly contain the specific fact asked "
        "(the right game, the right platform, the right attribute). "
        "A similar game, a sequel, or a different platform does NOT count. "
        "Questions about current or recent events, or about games/companies that do not appear in the documents, "
        "are NOT answerable from the documents. "
        "If the documents clearly contain the needed fact, confidence is 'high'. "
        "If they only partially cover it, use 'medium' or 'low'."
    )
    user = f"Question: {question}\n\nRetrieved documents:\n{json.dumps(retrieved_docs, indent=2, default=str)}"
    return llm_structured(system, user, EvaluationReport)

#### Game Web Search Tool

In [7]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Web search: searches the web with Tavily when the vector DB results aren't good enough
#    args:
#    - question: a question about game industry. 
#
#    Returns the search results, including each page's URL, so the answer can cite its source.

@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    res = tavily_client.search(query=question, max_results=4, include_answer=True)
    return {
        "tavily_answer": res.get("answer"),
        "results": [
            {"title": r.get("title"), "url": r.get("url"), "content": (r.get("content") or "")[:700]}
            for r in res.get("results", [])
        ],
    }

In [8]:
@tool
def recall_memory(query: str, n_results: int = 2) -> List[dict]:
    """
    Long-term memory: searches facts learned from earlier web searches.
    args:
    - query: a question about game industry.
    Returns remembered question/answer pairs together with the URL they came from.
    """
    total = memory_collection.count()
    if total == 0:
        return []
    res = memory_collection.query(query_texts=[query], n_results=min(n_results, total))
    out = []
    for i in range(len(res["ids"][0])):
        m = res["metadatas"][0][i]
        out.append({
            "source": "long_term_memory",
            "question": m["question"], "answer": m["answer"], "url": m["url"],
            "saved_at": m["saved_at"], "distance": round(res["distances"][0][i], 3),
        })
    return out


def remember(question: str, answer: str, url: str):
    """Save a web-sourced answer in long-term memory (upsert keyed by the question)."""
    mem_id = hashlib.md5(question.lower().strip().encode()).hexdigest()
    memory_collection.upsert(
        ids=[mem_id],
        documents=[f"Q: {question}\nA: {answer}"],
        metadatas=[{"question": question, "answer": answer, "url": url,
                    "saved_at": datetime.now(timezone.utc).strftime("%Y-%m-%d")}],
    )

### Agent

In [9]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
class AgentAnswer(BaseModel):
    answer: str = Field(description="Clear natural-language answer to the question")
    confidence: Literal["high", "medium", "low"]
    source_type: Literal["local_dataset", "web", "long_term_memory", "both"]
    sources: List[str] = Field(description="Citations: web URLs, or 'Local dataset:  (, id )'")


class Rewrite(BaseModel):
    standalone_question: str


# State carried through the workflow (lib.state_machine only keeps fields declared here)
class UdaState(TypedDict):
    session_id: str
    question: str
    history: List[dict]
    standalone: str
    local_docs: List[dict]
    memory_docs: List[dict]
    evaluation: Optional[dict]
    web: Optional[dict]
    answer: Optional[dict]

In [10]:
class UdaPlayAgent:
    def __init__(self, long_term_memory: bool = True):
        self.long_term_memory = long_term_memory
        self.memory = ShortTermMemory()          # conversation state: one list of Runs per session
        self.workflow = self._create_state_machine()

    # ------------------------------------------------------------------ workflow
    def _create_state_machine(self) -> StateMachine[UdaState]:
        machine = StateMachine[UdaState](UdaState)

        entry = EntryPoint[UdaState]()
        understand = Step[UdaState]("understand", self.s_understand)
        retrieve = Step[UdaState]("retrieve", self.s_retrieve)
        evaluate = Step[UdaState]("evaluate", self.s_evaluate)
        web_search = Step[UdaState]("web_search", self.s_web_search)
        answer = Step[UdaState]("answer", self.s_answer)
        remember_step = Step[UdaState]("remember", self.s_remember)
        end = Termination[UdaState]()

        machine.add_steps([entry, understand, retrieve, evaluate, web_search, answer, remember_step, end])

        machine.connect(entry, understand)
        machine.connect(understand, retrieve)
        machine.connect(retrieve, evaluate)

        def route_after_evaluate(state: UdaState):
            """Internal knowledge is good enough -> answer; otherwise fall back to the web."""
            e = state["evaluation"]
            if e["useful"] and e["confidence"] != "low":
                print("   decision: internal knowledge is good enough -> answer")
                return answer
            print("   decision: internal knowledge is NOT good enough -> web search")
            return web_search

        machine.connect(evaluate, [answer, web_search], route_after_evaluate)
        machine.connect(web_search, answer)
        machine.connect(answer, remember_step)
        machine.connect(remember_step, end)
        return machine

    # ------------------------------------------------------------------ steps
    def s_understand(self, state: UdaState) -> dict:
        print("\n>> STEP: understand")
        standalone = state["question"]
        if state["history"]:
            system = ("Rewrite the user's latest message as a standalone question about video games, "
                      "resolving pronouns and references (it, that game, its sequel...) using the conversation. "
                      "If it is already standalone, return it unchanged.")
            user = (f"Conversation so far:\n{json.dumps(state['history'][-6:], indent=2)}\n\n"
                    f"Latest message: {state['question']}")
            standalone = llm_structured(system, user, Rewrite).standalone_question
        print(f"   standalone question: {standalone}")
        return {"standalone": standalone}

    def s_retrieve(self, state: UdaState) -> dict:
        print("\n>> STEP: retrieve")
        local_docs = call_tool(retrieve_game, query=state["standalone"])
        for d in local_docs:
            print(f"     local : {d['Name']} | {d['Platform']} | {d['YearOfRelease']} | dist {d['distance']}")
        memory_docs = []
        if self.long_term_memory:
            memory_docs = call_tool(recall_memory, query=state["standalone"])
            for m in memory_docs:
                print(f"     memory: {m['question']} | dist {m['distance']}")
        return {"local_docs": local_docs, "memory_docs": memory_docs}

    def s_evaluate(self, state: UdaState) -> dict:
        print("\n>> STEP: evaluate")
        report = call_tool(evaluate_retrieval, question=state["standalone"],
                           retrieved_docs=state["local_docs"] + state["memory_docs"])
        print(f"   useful={report.useful} | confidence={report.confidence}")
        print(f"   reason: {report.description}")
        return {"evaluation": report.model_dump()}

    def s_web_search(self, state: UdaState) -> dict:
        print("\n>> STEP: web_search")
        try:
            web = call_tool(game_web_search, question=state["standalone"])
            for r in web["results"]:
                print(f"     web: {r['title'][:60]} | {r['url']}")
        except Exception as ex:
            print(f"   web search failed: {ex}")
            web = None
        return {"web": web}

    def s_answer(self, state: UdaState) -> dict:
        print("\n>> STEP: answer")
        evidence = {
            "local_dataset_results": state["local_docs"],
            "long_term_memory": state["memory_docs"],
            "retrieval_evaluation": state["evaluation"],
        }
        if state["web"]:
            evidence["web_search"] = state["web"]
        system = (
            "You are UdaPlay, an AI research assistant for the video game industry. "
            "Answer ONLY from the evidence provided; never invent facts. "
            "Use local dataset results when the evaluation says they are useful; otherwise rely on web results. "
            "If you combine sources, set source_type to 'both'. "
            "If the evidence is insufficient, say so plainly and set confidence to 'low'. "
            "Citations: for local data use 'Local dataset:  (, id )'; "
            "for web or memory facts use the exact URL from the evidence. "
            "Be concise and natural (1-3 sentences)."
        )
        user = (f"Conversation so far:\n{json.dumps(state['history'][-6:])}\n\n"
                f"Question: {state['standalone']}\n\nEvidence:\n{json.dumps(evidence, indent=2, default=str)}")
        ans = llm_structured(system, user, AgentAnswer)

        # guardrail: only keep URLs that really came from the evidence
        known = [r["url"] for r in (state["web"] or {}).get("results", [])] + [m["url"] for m in state["memory_docs"]]
        ans.sources = [s for s in ans.sources if (not s.startswith("http")) or s in known]
        if ans.source_type in ("web", "both") and known and not any(s.startswith("http") for s in ans.sources):
            ans.sources.append(known[0])
        return {"answer": ans.model_dump()}

    def s_remember(self, state: UdaState) -> dict:
        print("\n>> STEP: remember")
        ans = state["answer"]
        web_urls = [r["url"] for r in (state["web"] or {}).get("results", [])]
        cited_web = [s for s in ans["sources"] if s in web_urls]
        if self.long_term_memory and cited_web and ans["confidence"] != "low":
            remember(state["standalone"], ans["answer"], cited_web[0])
            print(f"   saved to long-term memory ({memory_collection.count()} memories)")
        else:
            print("   nothing to save")
        return {}

    # ------------------------------------------------------------------ public API
    def _history(self, session_id: str) -> List[dict]:
        """Rebuild the conversation from the Runs already stored for this session."""
        history = []
        for run in self.memory.get_all_objects(session_id):
            final = run.get_final_state()
            history.append({"role": "user", "content": final["question"]})
            history.append({"role": "assistant", "content": final["answer"]["answer"]})
        return history

    def ask(self, question: str, session_id: str = "default") -> AgentAnswer:
        print("\n" + "#" * 100)
        print(f"USER [{session_id}]: {question}")
        print("#" * 100)
        self.memory.create_session(session_id)
        initial_state: UdaState = {
            "session_id": session_id, "question": question, "history": self._history(session_id),
            "standalone": question, "local_docs": [], "memory_docs": [],
            "evaluation": None, "web": None, "answer": None,
        }
        run = self.workflow.run(initial_state)
        self.memory.add(run, session_id)           # conversation state for the next question
        self.report(run)
        return AgentAnswer(**run.get_final_state()["answer"])

    def get_session_runs(self, session_id: str = "default") -> List[Run]:
        return self.memory.get_all_objects(session_id)

    # ------------------------------------------------------------------ report
    def report(self, run: Run):
        s = run.get_final_state()
        a = s["answer"]
        print("\n" + "=" * 100)
        print("UDAPLAY REPORT")
        print("=" * 100)
        print(f"Question    : {s['question']}")
        if s["standalone"] != s["question"]:
            print(f"Interpreted : {s['standalone']}")
        print(f"Workflow    : {' -> '.join(snap.step_id for snap in run.snapshots)}")
        print(f"Answer      : {a['answer']}")
        print(f"Confidence  : {a['confidence']}")
        print(f"Source type : {a['source_type']}")
        print("Sources     :")
        for src in a["sources"]:
            print(f"   - {src}")
        print("\nStructured output (JSON):")
        print(json.dumps(a, indent=2, ensure_ascii=False))

In [11]:
# TODO: Invoke your agent
# - When was Pokémon Gold and Silver released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X released for PlayStation 5?

for _id in memory_collection.get()["ids"]:
    memory_collection.delete(ids=[_id])

agent = UdaPlayAgent()
a1 = agent.ask("When was Pokémon Gold and Silver released?")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given



####################################################################################################
USER [default]: When was Pokémon Gold and Silver released?
####################################################################################################
[StateMachine] Starting: __entry__

>> STEP: understand
   standalone question: When was Pokémon Gold and Silver released?
[StateMachine] Executing step: understand

>> STEP: retrieve
   [TOOL CALL] retrieve_game(query='When was Pokémon Gold and Silver released?')
     local : Pokémon Gold and Silver | Game Boy Color | 1999 | dist 0.237
     local : Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | dist 0.301
     local : Super Mario 64 | Nintendo 64 | 1996 | dist 0.459
   [TOOL CALL] recall_memory(query='When was Pokémon Gold and Silver released?')
[StateMachine] Executing step: retrieve

>> STEP: evaluate
   [TOOL CALL] evaluate_retrieval(question='When was Pokémon Gold and Silver released?', retrieved_docs=[{'Platform': '

In [12]:
a2 = agent.ask("Which one was the first 3D platformer Mario game?")


####################################################################################################
USER [default]: Which one was the first 3D platformer Mario game?
####################################################################################################
[StateMachine] Starting: __entry__

>> STEP: understand
   standalone question: Which game was the first 3D platformer in the Mario series?
[StateMachine] Executing step: understand

>> STEP: retrieve
   [TOOL CALL] retrieve_game(query='Which game was the first 3D platformer in the Mario series?')
     local : Super Mario 64 | Nintendo 64 | 1996 | dist 0.236
     local : Super Mario World | Super Nintendo Entertainment System (SNES) | 1990 | dist 0.278
     local : Mario Kart 8 Deluxe | Nintendo Switch | 2017 | dist 0.407
   [TOOL CALL] recall_memory(query='Which game was the first 3D platformer in the Mario series?')
[StateMachine] Executing step: retrieve

>> STEP: evaluate
   [TOOL CALL] evaluate_retrieval(question='Wh

In [13]:
a3 = agent.ask("Was Mortal Kombat X released for PlayStation 5?")


####################################################################################################
USER [default]: Was Mortal Kombat X released for PlayStation 5?
####################################################################################################
[StateMachine] Starting: __entry__

>> STEP: understand
   standalone question: Was Mortal Kombat X released for PlayStation 5?
[StateMachine] Executing step: understand

>> STEP: retrieve
   [TOOL CALL] retrieve_game(query='Was Mortal Kombat X released for PlayStation 5?')
     local : Marvel's Spider-Man 2 | PlayStation 5 | 2023 | dist 0.429
     local : Marvel's Spider-Man | PlayStation 4 | 2018 | dist 0.456
     local : Halo Infinite | Xbox Series X|S | 2021 | dist 0.457
   [TOOL CALL] recall_memory(query='Was Mortal Kombat X released for PlayStation 5?')
[StateMachine] Executing step: retrieve

>> STEP: evaluate
   [TOOL CALL] evaluate_retrieval(question='Was Mortal Kombat X released for PlayStation 5?', retrieved_docs

In [14]:
a4 = agent.ask("Who published it?")


####################################################################################################
USER [default]: Who published it?
####################################################################################################
[StateMachine] Starting: __entry__

>> STEP: understand
   standalone question: Who published Mortal Kombat X?
[StateMachine] Executing step: understand

>> STEP: retrieve
   [TOOL CALL] retrieve_game(query='Who published Mortal Kombat X?')
     local : Super Smash Bros. Melee | GameCube | 2001 | dist 0.443
     local : Halo Infinite | Xbox Series X|S | 2021 | dist 0.449
     local : Kinect Adventures! | Xbox 360 | 2010 | dist 0.472
   [TOOL CALL] recall_memory(query='Who published Mortal Kombat X?')


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


     memory: Was Mortal Kombat X released for PlayStation 5? | dist 0.333
[StateMachine] Executing step: retrieve

>> STEP: evaluate
   [TOOL CALL] evaluate_retrieval(question='Who published Mortal Kombat X?', retrieved_docs=[{'Platform': 'GameCube', 'Publisher': 'Nintendo', 'Name': 'Super Smas)
   useful=False | confidence=low
   reason: The retrieved documents do not contain any information about the publisher of Mortal Kombat X. They include details about other games, such as Super Smash Bros. Melee, Halo Infinite, and Kinect Adventures!, but none of these documents mention Mortal Kombat X or its publisher. The only relevant information found is about Mortal Kombat X's release on PlayStation 4 and its backward compatibility with PlayStation 5, but this does not answer the question regarding its publisher. Therefore, the documents are insufficient to respond to the query.
[StateMachine] Executing step: evaluate
   decision: internal knowledge is NOT good enough -> web search

>> STEP

In [15]:
a5 = agent.ask("What is Rockstar Games working on right now?")


####################################################################################################
USER [default]: What is Rockstar Games working on right now?
####################################################################################################
[StateMachine] Starting: __entry__

>> STEP: understand
   standalone question: What is Rockstar Games currently working on?
[StateMachine] Executing step: understand

>> STEP: retrieve
   [TOOL CALL] retrieve_game(query='What is Rockstar Games currently working on?')
     local : Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | dist 0.418
     local : Gran Turismo 5 | PlayStation 3 | 2010 | dist 0.44
     local : Halo Infinite | Xbox Series X|S | 2021 | dist 0.464
   [TOOL CALL] recall_memory(query='What is Rockstar Games currently working on?')
     memory: Who published Mortal Kombat X? | dist 0.543
     memory: Was Mortal Kombat X released for PlayStation 5? | dist 0.559
[StateMachine] Executing step: retrieve

>> ST

In [16]:
# Session Tracing

for i, run in enumerate(agent.get_session_runs(), 1):
    s = run.get_final_state()
    path = " -> ".join(snap.step_id for snap in run.snapshots)
    print(f"{i}. {s['question']!r}\n   {path}")

1. 'When was Pokémon Gold and Silver released?'
   __entry__ -> understand -> retrieve -> evaluate -> answer -> remember
2. 'Which one was the first 3D platformer Mario game?'
   __entry__ -> understand -> retrieve -> evaluate -> answer -> remember
3. 'Was Mortal Kombat X released for PlayStation 5?'
   __entry__ -> understand -> retrieve -> evaluate -> web_search -> answer -> remember
4. 'Who published it?'
   __entry__ -> understand -> retrieve -> evaluate -> web_search -> answer -> remember
5. 'What is Rockstar Games working on right now?'
   __entry__ -> understand -> retrieve -> evaluate -> web_search -> answer -> remember


### (Optional) Advanced — Stand Out

In [17]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

new_session_agent = UdaPlayAgent()
a6 = new_session_agent.ask("Was Mortal Kombat X released for PlayStation 5?", session_id="new-session")


####################################################################################################
USER [new-session]: Was Mortal Kombat X released for PlayStation 5?
####################################################################################################
[StateMachine] Starting: __entry__

>> STEP: understand
   standalone question: Was Mortal Kombat X released for PlayStation 5?
[StateMachine] Executing step: understand

>> STEP: retrieve
   [TOOL CALL] retrieve_game(query='Was Mortal Kombat X released for PlayStation 5?')
     local : Marvel's Spider-Man 2 | PlayStation 5 | 2023 | dist 0.429
     local : Marvel's Spider-Man | PlayStation 4 | 2018 | dist 0.456
     local : Halo Infinite | Xbox Series X|S | 2021 | dist 0.457
   [TOOL CALL] recall_memory(query='Was Mortal Kombat X released for PlayStation 5?')
     memory: Was Mortal Kombat X released for PlayStation 5? | dist 0.13
     memory: Who published Mortal Kombat X? | dist 0.341
[StateMachine] Executing step: re